# Ba thí nghiệm GPU còn lại - task 5, 6, 7

Gộp một kernel vì cả ba dùng **cùng một reader** (Qwen2.5-7B 4-bit). Nạp mô
hình một lần thay vì ba lần tiết kiệm khoảng 15 phút.

| # | Thí nghiệm | Ước tính | Trả lời câu hỏi gì |
|---|---|---|---|
| 5 | VimQA full (1.003 câu) | ~2.2h | Kết luận ở n=50 có đứng ở cỡ mẫu đầy đủ không |
| 6 | EXP-1 trần bộ dừng | ~0.5h | Bộ dừng hoàn hảo thì được thêm bao nhiêu F1 |
| 7 | EXP-2 tiếng Việt không dấu | ~0.5h | Gỡ dấu có phá bộ nén không |

**Thứ tự có chủ ý.** Task 6 và 7 chạy TRƯỚC vì mỗi cái chỉ 0.5h và trả lời
câu hỏi có/không rõ ràng. Task 5 chạy sau cùng vì nó dài nhất và có checkpoint
- hết giờ vẫn tiếp được ở lần chạy sau, còn 6/7 mà đứt giữa chừng thì mất trắng.

Kaggle cấp 30h GPU/tuần và tối đa **12h mỗi phiên**, nên cả ba vừa một phiên.

Trước khi chạy: **Settings → Accelerator → GPU T4**, **Internet → On**.

In [ ]:
!pip install -q llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -2


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ BẢNG CHÍNH n=200, READER 7B ══
# ~35 phút trên T4. run_eval.py ghi checkpoint sau MỖI câu nên hết giờ vẫn
# chạy tiếp được từ chỗ dở.
import subprocess, sys, os, time

READER = 'Qwen/Qwen2.5-7B-Instruct'
OUT = 'results/vimqa_200_7b.json'
os.makedirs('results', exist_ok=True)

def run_stream(cmd, logfile):
    """In output NGAY thay vì giữ tới lúc xong.

    7B ở 4-bit chạy hàng chục phút; capture_output nghĩa là ngồi nhìn màn hình
    trống, không biết treo hay đang chạy.
    """
    t0 = time.time()
    with open(logfile, 'w') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

if os.path.exists(OUT):
    print(f'[bỏ qua] đã có {OUT}')
else:
    run_stream([sys.executable, '-u', 'scripts/run_eval.py',
                '--dataset', 'vimqa', '--limit', '200',
                '--reader', 'hf', '--reader-model', READER, '--load-4bit',
                '--itercomp-llm', 'hf', '--scorer', 'dual',
                '--methods', 'llmlingua2,itercomp',
                '--out', OUT],
               'results/log_vimqa_200_7b.txt')

## Task 6 - EXP-1: trần của việc dừng đúng lúc

Nếu trần dưới 3 điểm F1 thì cải thiện bộ dừng không đáng làm, và cả hướng nghiên cứu đó dừng ở đây.

In [ ]:
# ══ TASK 6 - EXP-1 ══
import os, sys
READER = 'Qwen/Qwen2.5-7B-Instruct'
os.makedirs('results', exist_ok=True)

for ds in ('vimqa', 'musique'):
    out = f'results/exp1_oracle_{ds}_200_7b.json'
    if os.path.exists(out):
        print(f'[bỏ qua] {out} đã có'); continue
    print('=' * 62); print(f'### EXP-1 - {ds}  n=200'); print('=' * 62, flush=True)
    run_stream([sys.executable, '-u', 'research/exp1_oracle_stopping.py',
                '--dataset', ds, '--limit', '200',
                '--reader', 'hf', '--reader-model', READER, '--load-4bit',
                '--scorer', 'dual', '--percentile', '90', '--out', out],
               f'results/log_exp1_{ds}.txt')

## Task 7 - EXP-2: tiếng Việt KHÔNG DẤU

Lấp limitation báo cáo tự nêu. Mỗi chế độ chạy thêm nhánh gold-context để **tách nguyên nhân**: phần gold mất là lỗi reader, phần IterCOMP mất THÊM mới là lỗi bộ nén.

In [ ]:
# ══ TASK 7 - EXP-2 ══
out = 'results/exp2_undiacritised_200_7b.json'
if os.path.exists(out):
    print(f'[bỏ qua] {out} đã có')
else:
    print('=' * 62); print('### EXP-2 - tiếng Việt không dấu  n=200'); print('=' * 62, flush=True)
    run_stream([sys.executable, '-u', 'research/exp2_undiacritised.py',
                '--limit', '200',
                '--reader', 'hf', '--reader-model', READER, '--load-4bit',
                '--scorer', 'dual', '--percentile', '90', '--out', out],
               'results/log_exp2.txt')

## Task 5 - VimQA full 1.003 câu

Chạy CUỐI vì dài nhất. `run_eval.py` ghi checkpoint sau **mỗi câu**, nên hết
12h vẫn chạy lại tiếp được từ chỗ dở - với task này đó là tính năng bắt buộc,
không phải tiện lợi.

Ở n=1.003 ngưỡng phát hiện ghép cặp xuống khoảng **3.5 F1** (so với 17.5 ở
n=50), nên các ablation hiện nằm trong nhiễu mới có cơ hội tách được.

In [ ]:
# ══ TASK 5 - VimQA FULL ══
out = 'results/vimqa_full_7b.json'
if os.path.exists(out):
    print(f'[bỏ qua] {out} đã có')
else:
    print('=' * 62); print('### VimQA FULL  n=1003'); print('=' * 62, flush=True)
    run_stream([sys.executable, '-u', 'scripts/run_eval.py',
                '--dataset', 'vimqa',
                '--reader', 'hf', '--reader-model', READER, '--load-4bit',
                '--itercomp-llm', 'hf', '--scorer', 'dual',
                '--methods', 'raw,oracle,llmlingua2,itercomp',
                '--out', out],
               'results/log_vimqa_full.txt')

## Tải kết quả về

In [ ]:
import shutil, os
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z = shutil.make_archive(os.path.join(BASE, 'results_gpu_tasks'), 'zip', 'results')
print(f'✓ {z}  ({os.path.getsize(z)/1e6:.1f} MB)\n')
for f in sorted(os.listdir('results')):
    print(f'  {f}  ({os.path.getsize("results/"+f)/1024:.0f} KB)')